In [ ]:
# Run in terminal, not notebook:
# ----------------------------------------
# python3.10 -m venv ~/tfod_env
# source ~/tfod_env/bin/activate   # Linux/Mac
# ~/tfod_env\Scripts\activate      # Windows
# pip install --upgrade pip

# Install dependencies:
# pip install tensorflow==2.13.0 keras==2.13.1 \
#             tf-models-official==2.13.0 protobuf==3.20.* \
#             ml_dtypes==0.5.0 jax==0.4.13 jaxlib==0.4.13 \
#             pillow ipywidgets gdown

In [ ]:
import os
import sys
import shutil
import subprocess
import re
from pathlib import Path

HOMEFOLDER = '/Users/rubenhayrapetyan/Downloads/Code/FRC/machine-learning/2025-Coral_Detection_Training/Collab_training'
FINALOUTPUTFOLDER_DIRNAME = 'final_output'
FINALOUTPUTFOLDER = os.path.join(HOMEFOLDER, FINALOUTPUTFOLDER_DIRNAME)

print("Python version:", sys.version)
print("Home folder:", HOMEFOLDER)
print("Final output folder:", FINALOUTPUTFOLDER)

: 

In [ ]:
tmpModelPath = os.path.join(HOMEFOLDER, 'models')
if os.path.exists(tmpModelPath) and os.path.isdir(tmpModelPath):
    shutil.rmtree(tmpModelPath)
    print("Removed old models folder")

In [ ]:
# Clone repo
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/tensorflow/models', tmpModelPath
], check=True)

# Checkout specific commit
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 
                'ad1f7b56943998864db8f5db0706950e93bb7d81'], cwd=tmpModelPath, check=True)
subprocess.run(['git', 'checkout', 'ad1f7b56943998864db8f5db0706950e93bb7d81'], cwd=tmpModelPath, check=True)

print("Cloned TensorFlow models repo at commit ad1f7b5")

In [ ]:
protos_path = os.path.join(tmpModelPath, 'research', 'object_detection', 'protos')
subprocess.run(['protoc', '*.proto', f'--python_out={os.path.join(tmpModelPath,"research")}'], cwd=protos_path, shell=True)
print("Compiled proto files")

In [ ]:
setup_path = os.path.join(tmpModelPath, 'research', 'object_detection', 'packages', 'tf2', 'setup.py')
with open(setup_path) as f:
    s = f.read()
s = re.sub('tf-models-official>=2.5.1','tf-models-official==2.13.0', s)
with open(os.path.join(tmpModelPath, 'research', 'setup.py'), 'w') as f:
    f.write(s)

# Install object_detection API
subprocess.run([sys.executable, '-m', 'pip', 'install', os.path.join(tmpModelPath,'research')])
subprocess.run([sys.executable, '-m', 'pip', 'install', os.path.join(tmpModelPath,'research','slim')])

# Add to PYTHONPATH
os.environ['PYTHONPATH'] += f":{os.path.join(tmpModelPath,'research')}:{os.path.join(tmpModelPath,'research','slim')}"

# Test installation
test_file = os.path.join(tmpModelPath, 'research', 'object_detection', 'builders', 'model_builder_tf2_test.py')
subprocess.run([sys.executable, test_file])
print("Object Detection API installed and tested successfully")

In [ ]:
import requests

def download_dataset(drive_url, output_file):
    # Convert Google Drive sharing URL to direct download if needed
    if 'drive.google.com/file/d/' in drive_url:
        file_id = drive_url.split('/file/d/')[1].split('/')[0]
        drive_url = f'https://drive.google.com/uc?id={file_id}'
    r = requests.get(drive_url, allow_redirects=True)
    open(output_file, 'wb').write(r.content)
    print(f"Downloaded {output_file}")

# Example usage:
# dataset_zip = os.path.join(HOMEFOLDER,"dataset.zip")
# download_dataset("https://drive.google.com/file/d/xxx/view?usp=sharing", dataset_zip)

In [ ]:
import zipfile

dataset_zip = os.path.join(HOMEFOLDER,'dataset.zip')
with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
    zip_ref.extractall(HOMEFOLDER)
print("Dataset extracted")


In [ ]:
import tarfile

chosen_model = 'ssd-mobilenet-v2'
MODELS_CONFIG = {
    'ssd-mobilenet-v2': {
        'model_name': 'ssd_mobilenet_v2_320x320_coco17_tpu-8',
        'base_pipeline_file': 'limelight_ssd_mobilenet_v2_320x320_coco17_tpu-8.config',
        'pretrained_checkpoint': 'limelight_ssd_mobilenet_v2_320x320_coco17_tpu-8.tar.gz',
    },
}

model_name = MODELS_CONFIG[chosen_model]['model_name']
pretrained_checkpoint = MODELS_CONFIG[chosen_model]['pretrained_checkpoint']
base_pipeline_file = MODELS_CONFIG[chosen_model]['base_pipeline_file']

model_dir = os.path.join(tmpModelPath,'mymodel')
os.makedirs(model_dir, exist_ok=True)

# Download tar.gz
download_tar = f'https://downloads.limelightvision.io/models/{pretrained_checkpoint}'
r = requests.get(download_tar, allow_redirects=True)
tar_path = os.path.join(model_dir, pretrained_checkpoint)
open(tar_path, 'wb').write(r.content)

# Extract tar.gz
with tarfile.open(tar_path) as tar:
    tar.extractall(model_dir)
print("Pretrained model downloaded and extracted")


In [ ]:
from object_detection.utils import label_map_util

label_map_pbtxt_fname = os.path.join(HOMEFOLDER, 'dataset', 'label_map.pbtxt')  # replace with your actual path

def get_num_classes(pbtxt_fname):
    label_map = label_map_util.load_labelmap(pbtxt_fname)
    categories = label_map_util.convert_label_map_to_categories(label_map, max_num_classes=90, use_display_name=True)
    category_index = label_map_util.create_category_index(categories)
    return len(category_index.keys())

def get_classes(pbtxt_fname):
    label_map = label_map_util.load_labelmap(pbtxt_fname)
    categories = label_map_util.convert_label_map_to_categories(label_map, max_num_classes=90, use_display_name=True)
    category_index = label_map_util.create_category_index(categories)
    return [category['name'] for category in category_index.values()]

def create_label_file(filename, labels):
    with open(filename, 'w') as file:
        for label in labels:
            file.write(label + '\n')

num_classes = get_num_classes(label_map_pbtxt_fname)
classes = get_classes(label_map_pbtxt_fname)
create_label_file(os.path.join(HOMEFOLDER,"limelight_neural_detector_labels.txt"), classes)

print("Total classes:", num_classes)
print("Labels file created")


In [ ]:
pipeline_fname = os.path.join(model_dir, base_pipeline_file)
fine_tune_checkpoint = os.path.join(model_dir, model_name, 'checkpoint', 'ckpt-0')

# Replace paths in config file
with open(pipeline_fname) as f:
    s = f.read()

train_record_fname = os.path.join(HOMEFOLDER,'dataset','train.record')  # replace with your path
val_record_fname = os.path.join(HOMEFOLDER,'dataset','val.record')      # replace with your path

s = re.sub('fine_tune_checkpoint: ".*?"', f'fine_tune_checkpoint: "{fine_tune_checkpoint}"', s)
s = re.sub('input_path: ".*?train.*?"', f'input_path: "{train_record_fname}"', s)
s = re.sub('input_path: ".*?val.*?"', f'input_path: "{val_record_fname}"', s)
s = re.sub('label_map_path: ".*?"', f'label_map_path: "{label_map_pbtxt_fname}"', s)
s = re.sub('batch_size: [0-9]+', 'batch_size: 16', s)
s = re.sub('num_steps: [0-9]+', 'num_steps: 40000', s)

custom_pipeline_file = os.path.join(model_dir,'pipeline_file.config')
with open(custom_pipeline_file, 'w') as f:
    f.write(s)

print("Custom pipeline config saved")


In [ ]:
model_main_tf2_path = os.path.join(tmpModelPath,'research','object_detection','model_main_tf2.py')
model_dir_train = os.path.join(HOMEFOLDER,'training_progress')
os.makedirs(model_dir_train, exist_ok=True)

subprocess.run([
    sys.executable, model_main_tf2_path,
    f'--pipeline_config_path={custom_pipeline_file}',
    f'--model_dir={model_dir_train}',
    '--alsologtostderr',
    '--checkpoint_every_n=2000',
    '--num_train_steps=40000',
    '--num_workers=2',
    '--sample_1_of_n_eval_examples=1'
])


In [ ]:
exporter_path = os.path.join(tmpModelPath,'research','object_detection','export_tflite_graph_tf2.py')
os.makedirs(FINALOUTPUTFOLDER, exist_ok=True)

subprocess.run([
    sys.executable, exporter_path,
    f'--trained_checkpoint_dir={model_dir_train}',
    f'--output_directory={FINALOUTPUTFOLDER}',
    f'--pipeline_config_path={custom_pipeline_file}'
])

print("TFLite export completed. Saved in:", FINALOUTPUTFOLDER)

In [ ]:
import tensorflow as tf
import os
import io
from PIL import Image
import shutil

def extract_images_from_tfrecord(tfrecord_path, output_folder, num_samples=100):
    if os.path.exists(output_folder):
        shutil.rmtree(output_folder)
    os.makedirs(output_folder, exist_ok=True)

    saved_images = 0
    raw_dataset = tf.data.TFRecordDataset(tfrecord_path)
    for raw_record in raw_dataset.take(num_samples):
        example = tf.train.Example()
        example.ParseFromString(raw_record.numpy())
        image_data = example.features.feature['image/encoded'].bytes_list.value[0]
        image = Image.open(io.BytesIO(image_data))
        image.save(os.path.join(output_folder, f'image_{saved_images}.png'))
        saved_images += 1

    print(f"Extracted {saved_images} images to {output_folder}")

# Example usage:
train_record_fname = os.path.join(HOMEFOLDER,'dataset','train.record')  # replace with your path
extracted_sample_folder = os.path.join(HOMEFOLDER,'extracted_samples')

extract_images_from_tfrecord(train_record_fname, extracted_sample_folder, num_samples=100)


In [ ]:
import glob
import random

quant_image_list = glob.glob(os.path.join(extracted_sample_folder, '*.*'))
print("Number of images for quantization:", len(quant_image_list))

# Get model input size
model_path_32bit = os.path.join(FINALOUTPUTFOLDER, 'limelight_neural_detector_32bit.tflite')
interpreter = tf.lite.Interpreter(model_path=model_path_32bit)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
height = input_details[0]['shape'][1]
width = input_details[0]['shape'][2]

def representative_data_gen():
    for _ in range(300):
        pick_me = random.choice(quant_image_list)
        image = tf.io.read_file(pick_me)
        if pick_me.lower().endswith(('.jpg','.jpeg')):
            image = tf.io.decode_jpeg(image, channels=3)
        elif pick_me.lower().endswith('.png'):
            image = tf.io.decode_png(image, channels=3)
        elif pick_me.lower().endswith('.bmp'):
            image = tf.io.decode_bmp(image, channels=3)
        image = tf.image.resize(image, [height, width])
        image = tf.cast(image / 255.0, tf.float32)
        image = tf.expand_dims(image, 0)
        yield [image]


In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(FINALOUTPUTFOLDER,'saved_model'))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                       tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.float32

tflite_model_int8 = converter.convert()
tflite_model_int8_path = os.path.join(FINALOUTPUTFOLDER, 'limelight_neural_detector_8bit.tflite')
with open(tflite_model_int8_path, 'wb') as f:
    f.write(tflite_model_int8)

print("INT8 quantized model saved at:", tflite_model_int8_path)


In [ ]:
import zipfile

# Paths
labels_file = os.path.join(HOMEFOLDER, "limelight_neural_detector_labels.txt")
pipeline_config_file = os.path.join(HOMEFOLDER, "models", "mymodel", "pipeline_file.config")
zip_output_path = os.path.join(HOMEFOLDER, "limelight_detectors.zip")

# Make sure FINALOUTPUTFOLDER exists
if not os.path.exists(FINALOUTPUTFOLDER):
    os.makedirs(FINALOUTPUTFOLDER)

# Create ZIP
with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add TFLite models
    for tflite_file in ["limelight_neural_detector_32bit.tflite", "limelight_neural_detector_8bit.tflite"]:
        tflite_path = os.path.join(FINALOUTPUTFOLDER, tflite_file)
        if os.path.exists(tflite_path):
            zipf.write(tflite_path, arcname=tflite_file)
    
    # Add labels file
    if os.path.exists(labels_file):
        zipf.write(labels_file, arcname=os.path.basename(labels_file))
    
    # Add pipeline config file
    if os.path.exists(pipeline_config_file):
        zipf.write(pipeline_config_file, arcname=os.path.basename(pipeline_config_file))

print(f"All files packaged into: {zip_output_path}")
